# Занятие 8. Практика: анализ логов во времени и продуктовая витрина — АВТОРСКОЕ РЕШЕНИЕ

На прошлом занятии разобрали теорию: типы **`Timestamp`** и **`Timedelta`**, преобразование
текста в даты через **`pd.to_datetime()`** и аксессор **`.dt`**, интервалы между событиями,
группировки **`.groupby().agg()`** и сводные таблицы **`pd.pivot_table()`**.

Сегодня применяем это к логам интернет-магазина. Сначала **коротко вспомним инструменты**
на знакомом `orders.csv`

**План:**
1. повторение инструментов (на `orders.csv`);
2. Практика на данных `shop_orders.csv`.


In [56]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 220)
pd.set_option('display.precision', 2)

---
## Повторение инструментов

Вспомним приёмы на знакомом `orders.csv` (заказы из прошлого урока).

In [57]:
df = pd.read_csv('data/orders.csv', sep=';')
print('orders.csv:', df.shape)

orders.csv: (1500, 9)


### `pd.to_datetime()` и `.dt`

Текстовую дату превращаем в `Timestamp`, из него достаём компоненты через `.dt`:

In [58]:
ts = pd.to_datetime('2024-03-15 14:30')
print('год:', ts.year, '| час:', ts.hour, '| день недели:', ts.day_name())

год: 2024 | час: 14 | день недели: Friday


**Вывод:** у столбца-даты так же работают `.dt.year`, `.dt.hour`, `.dt.dayofweek`, `.dt.day_name()` — сразу для всей колонки.

### Интервалы между событиями

Разность двух дат — это `Timedelta`. Число дней получаем делением секунд на 86400:

In [59]:
b = pd.Timestamp('2024-05-01 12:30')
a = pd.Timestamp('2024-05-05 09:00')
print(a - b, '=', round((a - b).total_seconds() / 86400, 2), 'дня')

3 days 20:30:00 = 3.85 дня


**Вывод:** `.dt.total_seconds() / 86400` переводит интервал в дни с дробной частью.

### `groupby().agg()` — сводка по группам

Именованная агрегация: каждому столбцу отчёта — своя функция:

In [60]:
df.groupby('payment').agg(
    заказов=('order_id', 'count'),
    средний_чек=('amount', 'mean'),
).round(1)

,заказов,средний_чек
payment,,
Карта,821,4156.1
Наличные,149,3324.7
СБП,530,4099.1


**Вывод:** одним вызовом собрали число заказов и средний чек по каждому способу оплаты.

### `pd.pivot_table()` — двумерный отчёт

Одна переменная по строкам, другая по столбцам, в клетках — агрегат:

In [61]:
pd.pivot_table(df, index='city', columns='payment',
               values='order_id', aggfunc='count').fillna(0).astype(int).head()

payment,Карта,Наличные,СБП
city,,,
Екатеринбург,71,10,39
Казань,69,13,56
Москва,276,57,190
Нижний Новгород,56,10,30
Новосибирск,60,15,48


**Вывод:** получили таблицу «город × способ оплаты» с числом заказов в клетках.

---
## Практика

Ниже — **12 заданий** (максимум **30 баллов**). В каждом задании:
- **напишите код** в ячейке `# Ваш код` — код обязателен, ответ должен считаться им;
- **выведите результат** через `print()` или последней строкой.

Во многих заданиях есть пункт **Вывод:** — там нужно короткое предложение с трактовкой
результата (что получилось и что это значит).

### Датасет практики — `shop_orders.csv`

**1365 заказов** интернет-магазина, **15 колонок**:

| Столбец | Что это |
|---|---|
| `order_id` | номер заказа |
| `created_at` | дата и время оформления |
| `delivered_at` | дата и время доставки (у части пусто) |
| `customer_id` | клиент |
| `segment` | сегмент клиента: `New` / `Returning` / `VIP` |
| `channel` | канал привлечения: Ads / Organic / Email / Social / Referral |
| `device` | устройство: Mobile / Desktop / Tablet |
| `city` | город |
| `category` | категория товара |
| `payment` | способ оплаты |
| `amount` | сумма заказа, руб. |
| `quantity` | число позиций |
| `discount_pct` | скидка, % |
| `delivery_days` | срок доставки, дней (пусто, если не доставлен) |
| `rating` | оценка 1–5 (пусто, если без отзыва) |

Загружаем его и дальше работаем **с ним**:

In [62]:
df = pd.read_csv('data/shop_orders.csv', sep=';')
print('shop_orders.csv:', df.shape)
df.head()

shop_orders.csv: (1365, 15)


,order_id,created_at,delivered_at,customer_id,segment,channel,device,city,category,payment,amount,quantity,discount_pct,delivery_days,rating
0,1000,2024-01-01 07:48,2024-01-01 14:59,2231,New,Organic,Mobile,Москва,Книги,СБП,3321,5,0,0.3,4.0
1,1001,2024-01-01 10:32,NaN,2402,Returning,Social,Desktop,Екатеринбург,Электроника,СБП,46897,4,0,NaN,NaN
2,1002,2024-01-01 13:57,NaN,2055,Returning,Referral,Mobile,Ростов-на-Дону,Косметика,СБП,6252,4,0,NaN,NaN
3,1003,2024-01-01 19:56,2024-01-05 00:21,2433,Returning,Email,Mobile,Казань,Одежда,Карта,11586,3,0,3.2,5.0
4,1004,2024-01-02 13:18,2024-01-05 20:17,2261,Returning,Email,Desktop,Самара,Дом,СБП,3604,2,15,3.3,NaN


### <font color='DarkOrange'>Задание 1 [1 балл]</font>

Столбцы `created_at` и `delivered_at` пока текстовые. Преобразуйте **оба** в тип даты-времени через `pd.to_datetime()` и выведите их типы.

In [63]:
df['created_at'] = pd.to_datetime(df['created_at'])
df['delivered_at'] = pd.to_datetime(df['delivered_at'])
df[['created_at', 'delivered_at']].dtypes
# Ответ: оба столбца -> datetime64[ns]

created_at      datetime64[ns]
delivered_at    datetime64[ns]
dtype: object

### <font color='DarkOrange'>Задание 2 [2 балла]</font>

Из `created_at` извлеките два признака в новые столбцы: `час` (`.dt.hour`) и `день_недели` (`.dt.day_name()`). Выведите первые строки этих столбцов.

In [64]:
df['час'] = df['created_at'].dt.hour
df['день_недели'] = df['created_at'].dt.day_name()
df[['час', 'день_недели']].head()
# Ответ: столбцы час (0-23) и день_недели

,час,день_недели
0,7,Monday
1,10,Monday
2,13,Monday
3,19,Monday
4,13,Tuesday


### <font color='DarkOrange'>Задание 3 [2 балла]</font>

В какой **час суток** заказов больше всего? Найдите пиковый час и число заказов в нём (`value_counts`).

In [65]:
# заказы по часам — пиковый час стоит сверху (value_counts сортирует по убыванию)
df['час'].value_counts()
# Ответ: пик — 20:00 (155 заказов)

час
20    155
21    152
19    127
18    119
22     99
23     97
13     94
12     85
10     66
14     55
11     43
16     41
8      41
15     40
9      38
7      37
0      29
17     27
1      20
Name: count, dtype: int64

**Вывод:** Пик — 20:00 (155 заказов): заказывают вечером, после работы; днём и ночью заметно меньше.

### <font color='DarkOrange'>Задание 4 [3 балла]</font>

Постройте **матрицу активности по устройствам**: `pivot_table` с устройством (`device`) по строкам, часом (`час`) по столбцам и числом заказов в клетках (`aggfunc='count'`). По матрице видно, в какие часы заказывают с разных устройств.

In [66]:
df.pivot_table(index='device', columns='час',
               values='order_id', aggfunc='count').fillna(0).astype(int)
# Ответ: Desktop заказывает днём (9-18), Mobile и Tablet - вечером (18-23)

час,0,1,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
device,,,,,,,,,,,,,,,,,,,
Desktop,0,0,0,0,38,66,43,57,56,55,40,41,27,24,0,0,0,0,0
Mobile,29,20,37,41,0,0,0,28,38,0,0,0,0,95,98,99,99,62,68
Tablet,0,0,0,0,0,0,0,0,0,0,0,0,0,0,29,56,53,37,29


**Вывод:** По матрице видно: с Desktop заказывают в рабочие часы (9–18), а с Mobile и особенно Tablet — вечером (18–23).

### <font color='DarkOrange'>Задание 5 [2 балла]</font>

Посчитайте **средний чек** (`amount`) по сегментам клиентов (`segment`). Отсортируйте по возрастанию.

In [67]:
df.groupby('segment')['amount'].mean().round().sort_values()
# Ответ: New 5890, Returning 7885, VIP 11086

segment
New           5890.0
Returning     7885.0
VIP          11086.0
Name: amount, dtype: float64

**Вывод:** Средний чек растёт от New (5890) к VIP (11086) — почти вдвое: одна покупка VIP приносит заметно больше.

### <font color='DarkOrange'>Задание 6 [4 балла]</font>

Посчитайте **средний интервал между покупками одного клиента в разрезе сегмента**. Отсортируйте по `customer_id` и `created_at`, возьмите `.groupby('customer_id')['created_at'].diff()`, переведите в дни, затем усредните по `segment`.

In [68]:
ds = df.sort_values(['customer_id', 'created_at'])
ds['gap'] = ds.groupby('customer_id')['created_at'].diff().dt.total_seconds() / 86400
ds.groupby('segment')['gap'].mean().round(1)
# Ответ: New 124.3, Returning 75.0, VIP 33.1 дня

segment
New          124.3
Returning     75.0
VIP           33.1
Name: gap, dtype: float64

**Вывод:** VIP возвращаются в среднем раз в 33 дня — почти вчетверо чаще, чем New (124 дня): самый активный и лояльный сегмент.

### <font color='DarkOrange'>Задание 7 [3 балла]</font>

Соберите **продуктовую витрину** по категориям через `groupby('category').agg(...)`: `заказов` (count), `средний_чек` (mean `amount`), `выручка` (sum `amount`), `клиентов` (nunique `customer_id`). Отсортируйте по выручке по убыванию.

In [69]:
showcase = df.groupby('category').agg(
    заказов=('order_id', 'count'),
    средний_чек=('amount', 'mean'),
    выручка=('amount', 'sum'),
    клиентов=('customer_id', 'nunique'),
).round(0)
showcase.sort_values('выручка', ascending=False)
# Ответ: лидер по выручке - Электроника (~4.93 млн)

,заказов,средний_чек,выручка,клиентов
category,,,,
Электроника,178,27719.0,4934012,139
Одежда,277,7208.0,1996677,200
Спорт,140,9534.0,1334803,116
Дом,213,5342.0,1137923,160
Косметика,201,4290.0,862369,153
Игрушки,144,3275.0,471640,116
Книги,212,1946.0,412487,161


**Вывод:** Больше всего выручки даёт Электроника (~4.9 млн) — не за счёт объёма, а за счёт самого высокого чека: она обгоняет более массовые Одежду и Книги.

### <font color='DarkOrange'>Задание 8 [3 балла]</font>

Найдите **самый активный день недели** — в какой день заказов больше всего. На выбор: постройте `pivot_table` день недели × час и просуммируйте по строкам, **или** сгруппируйте по `день_недели` и посчитайте заказы.

In [70]:
df.groupby('день_недели')['order_id'].count().sort_values(ascending=False)
# Ответ: Sunday - 214 заказов (отрыв небольшой)

день_недели
Sunday       214
Tuesday      211
Thursday     197
Monday       188
Saturday     186
Wednesday    186
Friday       183
Name: order_id, dtype: int64

**Вывод:** Формально лидирует воскресенье (214), но отрыв от других дней небольшой — по дням недели активность довольно ровная.

### <font color='DarkOrange'>Задание 9 [3 балла]</font>

Посчитайте **выручку по каналам** (`channel`, сумма `amount`) и отдельно — **долю выручки от VIP** в каждом канале (выручка VIP в канале / вся выручка канала). Какой канал приводит самых ценных клиентов?

In [71]:
revenue = df.groupby('channel')['amount'].sum()
vip = df[df['segment'] == 'VIP'].groupby('channel')['amount'].sum()
pd.DataFrame({'выручка': revenue, 'доля_VIP': (vip / revenue).round(2)}).sort_values('выручка', ascending=False)
# Ответ: по выручке лидер Ads; по доле VIP - Ads 0.36 и Referral 0.35

,выручка,доля_VIP
channel,,
Ads,3300460,0.36
Organic,2663515,0.33
Social,1793573,0.26
Email,1766567,0.33
Referral,1625796,0.35


**Вывод:** Ads даёт и самую большую выручку (3.3 млн), и максимальную долю VIP (36%); Referral приводит почти столько же VIP (35%) при меньшем объёме — оба канала «премиальные».

### <font color='DarkOrange'>Задание 10 [2 балла]</font>

Оцените **эффективность скидок**. Создайте признак `со_скидкой` (`discount_pct > 0`) и сравните для заказов со скидкой и без: средний чек (`amount`), среднюю корзину (`quantity`) и число заказов.

In [72]:
df['со_скидкой'] = df['discount_pct'] > 0
df.groupby('со_скидкой').agg(
    средний_чек=('amount', 'mean'),
    средняя_корзина=('quantity', 'mean'),
    заказов=('order_id', 'count'),
).round(2)
# Ответ: со скидкой чек ниже (7383 vs 8465), корзина та же (~2.05)

,средний_чек,средняя_корзина,заказов
со_скидкой,,,
False,8464.80,2.05,991
True,7383.14,2.06,374


**Вывод:** Скидки не увеличивают корзину (по ~2.05 позиции и там, и там), но снижают средний чек (7383 против 8465) — то есть работают как прямая потеря маржи, а не драйвер объёма.

### <font color='DarkOrange'>Задание 11 [3 балла]</font>

Разбейте сутки на **4 части** и сравните их. Создайте столбец `время_суток` из `час`: ночь (0–5), утро (6–11), день (12–17), вечер (18–23). Посчитайте по каждой части число заказов и средний чек (`groupby().agg()`). Когда заказов больше всего, а когда средний чек выше?

In [73]:
def time_of_day(h):
    if h < 6:  return 'ночь'
    if h < 12: return 'утро'
    if h < 18: return 'день'
    return 'вечер'

df['время_суток'] = df['час'].apply(time_of_day)
df.groupby('время_суток').agg(
    заказов=('order_id', 'count'),
    средний_чек=('amount', 'mean'),
).round(0).sort_values('заказов', ascending=False)
# Ответ: заказов больше всего вечером (749); чек выше ночью (~9975) и утром (~9306)

,заказов,средний_чек
время_суток,,
вечер,749,7916.0
день,342,7715.0
утро,225,9306.0
ночь,49,9975.0


**Вывод:** Больше всего заказов вечером (749), но средний чек там НЕ самый высокий: ночью и утром заказов мало, зато чек заметно выше (~9975 и ~9306) — редкие ночные заказы крупнее вечерних.

### <font color='DarkOrange'>Задание 12 [2 балла]</font>

Аналитика. Посчитайте **выручку на одного клиента по сегментам** (сумма `amount` / число уникальных клиентов). Опираясь на это и выводы заданий 5, 6 и 9, предложите, на какой сегмент и через какой канал выгоднее тратить маркетинг. Обоснуйте в выводе.

In [74]:
(df.groupby('segment')['amount'].sum()
 / df.groupby('segment')['customer_id'].nunique()).round(0).sort_values(ascending=False)
# Ответ: New 9033, Returning 28295, VIP 105794 руб/клиент

segment
VIP          105794.0
Returning     28295.0
New            9033.0
dtype: float64

**Вывод:** Один VIP приносит ~105 800 руб против 9 000 у New — на порядок больше, и возвращается втрое чаще. Значит выгоднее вкладываться в удержание и привлечение VIP-подобных клиентов, а самый «премиальный» канал для этого — Ads (и Referral): именно они дают наибольшую долю VIP-выручки.